In [14]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [15]:
#hide
from fastbook import *

# Классификация изображений

Теперь, когда вы понимаете, что такое глубокое обучение, для чего оно нужно и как создавать и развертывать модели, пришло время углубиться! В идеальном мире специалистам по глубокому обучению не пришлось бы знать каждую деталь работы алгоритмов «под капотом»… Но пока мы живем не в идеальном мире. На деле, чтобы ваша модель работала по-настоящему эффективно и стабильно, нужно правильно настроить множество деталей и тщательно их проверить. Этот процесс требует возможности заглянуть внутрь вашей нейросети во время обучения и предсказаний, выявить потенциальные проблемы и знать, как их исправить.  

Итак, далее в книге мы погрузимся в механику глубокого обучения. Как устроены архитектуры моделей компьютерного зрения, NLP (Natural Language Processing, обработки естественного языка), табличных данных и других типов? Как создать архитектуру, подходящую под задачи вашей предметной области? Как добиться наилучших результатов в процессе обучения? Как ускорить работу? Что нужно менять при изменении наборов данных?  

Мы начнем с повторения базовых примеров, которые рассматривали в первой главе, но сделаем две вещи:  

- Улучшим их.  
- Применим к большему разнообразию типов данных.  

Для этого нам предстоит изучить все элементы пазла глубокого обучения. Сюда входят разные типы слоев, методы регуляризации, оптимизаторы, способы комбинирования слоев в архитектуры, техники разметки данных и многое другое. Однако мы не будем вываливать на вас все сразу — мы будем вводить их постепенно, по мере необходимости, для решения конкретных задач в рамках наших проектов.

## От собак и кошек к породам питомцев

В нашей самой первой модели мы научились классифицировать собак и кошек. Еще несколько лет назад эта задача считалась очень сложной — но сегодня она стала слишком простой! На этом примере мы не сможем показать все нюансы обучения моделей, поскольку получаем практически идеальный результат, не заботясь о деталях. Однако оказывается, что тот же набор данных позволяет нам работать над гораздо более сложной задачей: определить породу питомца на каждом изображении.

В главе \<\<chapter_intro\>\> мы представили приложения как уже решенные задачи. Но в реальной жизни все устроено иначе. Мы начинаем с какого-то набора данных, о котором ничего не знаем. Затем нам нужно разобраться, как он устроен, как извлечь нужные данные и как эти данные выглядят. В оставшейся части книги мы покажем, как решать эти задачи на практике, включая все промежуточные шаги, необходимые для понимания данных, с которыми вы работаете, и тестирования модели по мере ее создания.

Мы уже загрузили набор данных Pet и можем получить путь к нему, используя тот же код, что и в \<\<chapter_intro\>\>:

In [16]:
from fastai.vision.all import *
path = untar_data(URLs.PETS)

Чтобы понять, как извлекать породу каждого питомца из изображений, нам нужно разобраться в структуре данных. Такие детали организации данных — ключевая часть головоломки глубокого обучения. Обычно данные предоставляются в одном из двух форматов:  

- **Отдельные файлы**, представляющие элементы данных (например, текстовые документы или изображения), которые могут быть организованы в папки или иметь имена, содержащие информацию об этих элементах  
- **Табличные данные** (например, в формате CSV), где каждая строка представляет элемент и может включать имена файлов, связывающие данные таблицы с другими форматами (текстовыми документами или изображениями)  

Существуют исключения из этих правил — особенно в таких областях, как геномика, где могут встречаться бинарные форматы баз данных или даже сетевые потоки, — но в подавляющем большинстве случаев наборы данных используют комбинацию этих двух форматов.  

Чтобы посмотреть содержимое нашего набора данных, можно использовать метод `ls`:

In [17]:
#hide
Path.BASE_PATH = path

In [18]:
path.ls()

(#2) [Path('annotations'),Path('images')]

Мы видим, что этот набор данных содержит директории *images* (изображения) и *annotations* (аннотации). Согласно [сайту](https://www.robots.ox.ac.uk/~vgg/data/pets/) датасета, директория *annotations* содержит информацию о местоположении питомцев на изображениях, а не об их породах. В этой главе мы занимаемся классификацией, а не локализацией — то есть нас интересует, *что* изображено, а не *где*. Поэтому пока мы проигнорируем директорию *annotations*. Давайте посмотрим содержимое директории *images*:

In [19]:
(path/"images").ls()

(#7393) [Path('images/Siamese_183.jpg'),Path('images/Bombay_90.jpg'),Path('images/Ragdoll_122.jpg'),Path('images/keeshond_78.jpg'),Path('images/havanese_3.jpg'),Path('images/american_pit_bull_terrier_70.jpg'),Path('images/basset_hound_62.jpg'),Path('images/Sphynx_121.jpg'),Path('images/Ragdoll_170.jpg'),Path('images/english_cocker_spaniel_14.jpg'),Path('images/newfoundland_8.jpg'),Path('images/shiba_inu_38.jpg'),Path('images/english_setter_189.jpg'),Path('images/Birman_105.jpg'),Path('images/german_shorthaired_173.jpg'),Path('images/British_Shorthair_99.jpg'),Path('images/wheaten_terrier_25.jpg'),Path('images/chihuahua_99.jpg'),Path('images/german_shorthaired_190.jpg'),Path('images/chihuahua_33.jpg')...]

Большинство функций и методов в fastai, возвращающих коллекции, используют класс `L`. Его можно рассматривать как улучшенную версию стандартного Python-`list` с дополнительными удобствами для частых операций. Например, при отображении объекта этого класса в ноутбуке он появляется в специальном формате. Первым указывается количество элементов с префиксом `#`. Также в выводе присутствует многоточие — это означает, что отображаются только первые элементы, что удобно, ведь 7000+ имён файлов на экране были бы излишними!

Анализируя имена файлов, можно заметить их структуру: каждое название содержит породу питомца, затем подчёркивание (`_`), номер и расширение файла. Нам нужно написать код для извлечения породы из объекта `Path`. Jupyter Notebook упрощает эту задачу, позволяя постепенно разрабатывать решение и затем применять его ко всему датасету. Важно избегать излишних допущений: например, некоторые названия пород состоят из нескольких слов, поэтому нельзя просто разделить строку по первому символу `_`. Для тестирования возьмём одно из имён файлов:

In [20]:
fname = (path/"images").ls()[0]

Наиболее мощный и гибкий способ извлечения информации из строк — *регулярные выражения* (regular expressions, regex). Регулярное выражение — это специальная строка, написанная на языке regex, которая задаёт правило для проверки соответствия другой строки определённому шаблону, а также для извлечения конкретных частей из этой строки.

В нашем случае нужно регулярное выражение, которое извлекает породу питомца из имени файла.

У нас нет возможности дать полный урок по регулярным выражениям здесь, но в интернете есть множество отличных руководств, и многие из вас уже знакомы с этим замечательным инструментом. Если нет — это прекрасный момент, чтобы это исправить! Регулярные выражения — один из самых полезных инструментов в нашем арсенале, и многие студенты отмечают, что их изучение было одним из самых интересных этапов. Так что смело ищите в Google «регулярные выражения tutorial» и возвращайтесь, когда освоите основы. На [сайте книги](https://book.fast.ai/) также есть список наших любимых ресурсов.

> a: Регулярные выражения не только невероятно полезны, но и имеют интересные корни. Они «регулярные», потому что изначально были примерами «регулярного» языка — нижней ступени в иерархии Хомского, классификации грамматик, разработанной лингвистом Ноамом Хомским (Noam Chomsky), автором работы «Синтаксические структуры», где он исследовал формальные грамматики, лежащие в основе человеческого языка. Это одна из прелестей программирования: инструмент, которым вы пользуетесь ежедневно, может иметь происхождение из совершенно неожиданной области.

При написании регулярного выражения лучше всего начать с проверки на одном примере. Используем метод `findall`, чтобы протестировать регулярное выражение на имени файла из объекта `fname`:

In [21]:
re.findall(r'(.+)_\d+.jpg$', fname.name)

['Siamese']

Данное регулярное выражение извлекает все символы до последнего подчеркивания, при условии что последующие символы представляют цифры и расширение JPEG.

Теперь, когда мы подтвердили, что регулярное выражение работает на примере, давайте используем его для разметки всего набора данных. В fastai есть множество классов для помощи в разметке. Для разметки с помощью регулярных выражений мы можем использовать класс `RegexLabeller`. В этом примере мы используем API блоков данных, который видели в \<\<chapter_production\>\> (на самом деле мы почти всегда используем API блоков данных — он гораздо более гибкий, чем простые фабричные методы из \<\<chapter_intro\>\>):

In [22]:
pets = DataBlock(blocks = (ImageBlock, CategoryBlock),
                 get_items=get_image_files,
                 splitter=RandomSplitter(seed=42),
                 get_y=using_attr(RegexLabeller(r'(.+)_\d+.jpg$'), 'name'),
                 item_tfms=Resize(460),
                 batch_tfms=aug_transforms(size=224, min_scale=0.75))
dls = pets.dataloaders(path/"images")

Одна важная часть этого вызова `DataBlock`, которую мы раньше не видели, находится в этих двух строках:

```python
item_tfms=Resize(460),
batch_tfms=aug_transforms(size=224, min_scale=0.75)
```

Эти строки реализуют стратегию аугментации данных fastai, которую мы называем *предварительным масштабированием* (presizing). Предварительное масштабирование — это особый способ аугментации изображений, разработанный для минимизации потери данных при сохранении высокой производительности.

## Предварительное масштабирование (Presizing)

Нам нужно, чтобы изображения имели одинаковые размеры, чтобы их можно было объединить в тензоры для передачи на GPU. Также мы хотим минимизировать количество отдельных операций аугментации. Требования к производительности подсказывают, что по возможности следует комбинировать преобразования аугментации в меньшее количество операций (чтобы сократить вычисления и потери от сжатия) и приводить изображения к единому размеру (для более эффективной обработки на GPU).

Проблема в том, что если выполнять различные распространённые преобразования аугментации после уменьшения до целевого размера, это может привести к появлению ложных пустых областей, ухудшению качества данных или тому и другому одновременно. Например, поворот изображения на 45 градусов заполняет угловые области новыми границами пустоты, что не даст модели полезной информации. Многие операции поворота и масштабирования требуют интерполяции для создания пикселей. Эти интерполированные пиксели основаны на исходных данных, но всё равно имеют более низкое качество.

Чтобы обойти эти проблемы, предварительное масштабирование (presizing) использует две стратегии, показанные в <<presizing>>:

1. Изменение размеров изображений до относительно «больших» значений — то есть значительно больше целевых размеров для обучения.  
1. Объединение всех стандартных операций аугментации (включая изменение до конечного целевого размера) в одну операцию, которая выполняется на GPU единожды в конце обработки, вместо последовательного выполнения операций с многократной интерполяцией.

Первый шаг (изменение размера) создаёт изображения с достаточным запасом, чтобы дальнейшие преобразования аугментации можно было применять к внутренним областям без образования пустых зон. Это преобразование работает за счёт масштабирования до квадрата с использованием большого размера обрезки (crop size). В обучающем наборе область обрезки выбирается случайным образом, а её размер подбирается так, чтобы покрыть всю ширину или высоту изображения — в зависимости от того, что меньше.

На втором шаге GPU используется для всей аугментации данных, и все потенциально разрушительные операции выполняются вместе, с единственной интерполяцией в конце.

<img alt="Presizing on the training set" width="600" caption="Presizing on the training set" id="presizing" src="https://github.com/Wansmer/fastbook/blob/ru/images/att_00060.png?raw=1">

На этом изображении показаны два шага:

1. *Обрезка по полной ширине или высоте*: Это находится в `item_tfms`, поэтому применяется к каждому отдельному изображению перед его копированием на GPU. Это используется для обеспечения одинакового размера всех изображений. В обучающем наборе область обрезки выбирается случайным образом. В валидационном наборе всегда выбирается центральный квадрат изображения.
2. *Случайная обрезка и аугментация*: Это находится в `batch_tfms`, поэтому применяется ко всей партии сразу на GPU, что делает это быстрым. В валидационном наборе здесь выполняется только изменение размера до окончательного размера, необходимого для модели. В обучающем наборе сначала выполняется случайная обрезка и любые другие аугментации.

Чтобы реализовать этот процесс в fastai, вы используете `Resize` в качестве трансформации элемента с большим размером и `RandomResizedCrop` в качестве трансформации партии с меньшим размером. `RandomResizedCrop` будет добавлен автоматически, если вы включите параметр `min_scale` в вашу функцию `aug_transforms`, как это было сделано в вызове `DataBlock` в предыдущем разделе. В качестве альтернативы вы можете использовать `pad` или `squish` вместо `crop` (по умолчанию) для начального `Resize`.

\<\<interpolations\>\> показывает разницу между изображением, которое было увеличено, интерполировано, повернуто, а затем снова интерполировано (что является подходом, используемым всеми другими библиотеками глубокого обучения), показанным здесь справа, и изображением, которое было увеличено и повернуто как одно действие, а затем интерполировано только один раз слева (подход fastai), показанным здесь слева.

In [23]:
#hide_input
#id interpolations
#caption A comparison of fastai's data augmentation strategy (left) and the traditional approach (right).
dblock1 = DataBlock(blocks=(ImageBlock(), CategoryBlock()),
                   get_y=parent_label,
                   item_tfms=Resize(460))
# Place an image in the 'images/grizzly.jpg' subfolder where this notebook is located before running this
dls1 = dblock1.dataloaders([(Path.cwd()/'images'/'grizzly.jpg')]*100, bs=8)
dls1.train.get_idxs = lambda: Inf.ones
x,y = dls1.valid.one_batch()
_,axs = subplots(1, 2)

x1 = TensorImage(x.clone())
x1 = x1.affine_coord(sz=224)
x1 = x1.rotate(draw=30, p=1.)
x1 = x1.zoom(draw=1.2, p=1.)
x1 = x1.warp(draw_x=-0.2, draw_y=0.2, p=1.)

tfms = setup_aug_tfms([Rotate(draw=30, p=1, size=224), Zoom(draw=1.2, p=1., size=224),
                       Warp(draw_x=-0.2, draw_y=0.2, p=1., size=224)])
x = Pipeline(tfms)(x)
#x.affine_coord(coord_tfm=coord_tfm, sz=size, mode=mode, pad_mode=pad_mode)
TensorImage(x[0]).show(ctx=axs[0])
TensorImage(x1[0]).show(ctx=axs[1]);

FileNotFoundError: [Errno 2] No such file or directory: '/content/images/grizzly.jpg'

Вы можете видеть, что изображение справа менее четкое и имеет артефакты отражающей обрезки в нижнем левом углу; также трава в верхнем левом углу полностью исчезла. На практике мы обнаруживаем, что использование предварительного изменения размера значительно улучшает точность моделей и часто также приводит к увеличению скорости.

Библиотека fastai также предоставляет простые способы проверить, правильно ли выглядят ваши данные перед обучением модели, что является крайне важным шагом. Мы рассмотрим это далее.

### Проверка и отладка DataBlock

Мы никогда не можем просто предполагать, что наш код работает идеально. Написание `DataBlock` похоже на создание чертежа. Вы получите сообщение об ошибке, если у вас есть синтаксическая ошибка где-то в коде, но у вас нет гарантии, что ваш шаблон будет работать с вашим источником данных так, как вы намеревались. Поэтому перед обучением модели вы всегда должны проверять свои данные. Вы можете сделать это, используя метод `show_batch`:

In [ ]:
dls.show_batch(nrows=1, ncols=3)

Посмотрите на каждое изображение и проверьте, что каждое из них, кажется, имеет правильную метку для этой породы питомца. Часто ученые-данные работают с данными, с которыми они не так знакомы, как эксперты в области: например, я на самом деле не знаю, что представляют собой многие из этих пород питомцев. Поскольку я не являюсь экспертом по породам питомцев, я бы в этот момент использовал Google Images, чтобы поискать несколько из этих пород и убедиться, что изображения выглядят похоже на то, что я вижу в этом выводе.

Если вы допустили ошибку при создании вашего `DataBlock`, очень вероятно, что вы не увидите ее до этого шага. Чтобы отладить это, мы рекомендуем использовать метод `summary`. Он попытается создать партию из источника, который вы ему предоставите, с множеством деталей. Также, если он не удастся, вы увидите точно, на каком этапе произошла ошибка, и библиотека попытается дать вам некоторую помощь. Например, одной из распространенных ошибок является забывание использовать трансформацию `Resize`, в результате чего вы получаете изображения разных размеров и не можете объединить их в партию. Вот как будет выглядеть сводка в этом случае (обратите внимание, что точный текст мог измениться с момента написания, но это даст вам представление):

In [ ]:
#hide_output
pets1 = DataBlock(blocks = (ImageBlock, CategoryBlock),
                 get_items=get_image_files,
                 splitter=RandomSplitter(seed=42),
                 get_y=using_attr(RegexLabeller(r'(.+)_\d+.jpg$'), 'name'))
pets1.summary(path/"images")

```
Setting-up type transforms pipelines
Collecting items from /home/sgugger/.fastai/data/oxford-iiit-pet/images
Found 7390 items
2 datasets of sizes 5912,1478
Setting up Pipeline: PILBase.create
Setting up Pipeline: partial -> Categorize

Building one sample
  Pipeline: PILBase.create
    starting from
      /home/sgugger/.fastai/data/oxford-iiit-pet/images/american_bulldog_83.jpg
    applying PILBase.create gives
      PILImage mode=RGB size=375x500
  Pipeline: partial -> Categorize
    starting from
      /home/sgugger/.fastai/data/oxford-iiit-pet/images/american_bulldog_83.jpg
    applying partial gives
      american_bulldog
    applying Categorize gives
      TensorCategory(12)

Final sample: (PILImage mode=RGB size=375x500, TensorCategory(12))

Setting up after_item: Pipeline: ToTensor
Setting up before_batch: Pipeline:
Setting up after_batch: Pipeline: IntToFloatTensor

Building one batch
Applying item_tfms to the first sample:
  Pipeline: ToTensor
    starting from
      (PILImage mode=RGB size=375x500, TensorCategory(12))
    applying ToTensor gives
      (TensorImage of size 3x500x375, TensorCategory(12))

Adding the next 3 samples

No before_batch transform to apply

Collating items in a batch
Error! It's not possible to collate your items in a batch
Could not collate the 0-th members of your tuples because got the following
shapes:
torch.Size([3, 500, 375]),torch.Size([3, 375, 500]),torch.Size([3, 333, 500]),
torch.Size([3, 375, 500])
```

Вы можете точно увидеть, как мы собрали данные и разделили их, как мы перешли от имени файла к *образцу* (кортеж (изображение, категория)), затем какие трансформации элементов были применены и как не удалось объединить эти образцы в партию (из-за различных форм).

Как только вы думаете, что ваши данные выглядят правильно, мы обычно рекомендуем следующим шагом использовать их для обучения простой модели. Мы часто видим, как люди откладывают обучение фактической модели слишком надолго. В результате они не узнают, каковы их базовые результаты. Возможно, вашей задаче не требуется много сложной специфической для области инженерии. Или, возможно, данные вообще не обучают модель. Это то, что вы хотите узнать как можно скорее. Для этого первоначального теста мы будем использовать ту же простую модель, которую использовали в \<\<chapter_intro\>\>:

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(2)

Как мы уже кратко обсуждали ранее, таблица, отображаемая при обучении модели, показывает нам результаты после каждой эпохи обучения. Помните, что эпоха — это один полный проход по всем изображениям в данных. Отображаемые столбцы — это средняя потеря по элементам обучающего набора, потеря на валидационном наборе и любые метрики, которые мы запросили — в данном случае, уровень ошибок.

Помните, что *потеря* — это любая функция, которую мы решили использовать для оптимизации параметров нашей модели. Но мы на самом деле не сказали fastai, какую функцию потерь мы хотим использовать. Так что же он делает? fastai, как правило, пытается выбрать подходящую функцию потерь в зависимости от того, какой тип данных и модели вы используете. В данном случае у нас есть данные изображений и категориальный результат, поэтому fastai по умолчанию использует *потерю кросс-энтропии*.

## Потеря Кросс-Энтропии

*Потеря кросс-энтропии* — это функция потерь, которая похожа на ту, которую мы использовали в предыдущей главе, но (как мы увидим) имеет два преимущества:

- Она работает даже тогда, когда наша зависимая переменная имеет более двух категорий.
- Она приводит к более быстрому и надежному обучению.

Чтобы понять, как работает потеря кросс-энтропии для зависимых переменных с более чем двумя категориями, нам сначала нужно понять, как выглядят фактические данные и активации, которые видит функция потерь.

### Просмотр активаций и меток

Давайте посмотрим на активации нашей модели. Чтобы получить партию реальных данных из наших `DataLoaders`, мы можем использовать метод `one_batch`:

In [ ]:
x,y = dls.one_batch()

Как вы видите, это возвращает зависимые и независимые переменные в виде мини-партии. Давайте посмотрим, что на самом деле содержится в нашей зависимой переменной:

In [ ]:
y

Наш размер партии составляет 64, поэтому в этом тензоре у нас 64 строки. Каждая строка — это одно целое число от 0 до 36, представляющее наши 37 возможных пород питомцев. Мы можем просмотреть предсказания (то есть активации последнего слоя нашей нейронной сети), используя `Learner.get_preds`. Эта функция принимает либо индекс набора данных (0 для обучающего и 1 для валидационного), либо итератор партий. Таким образом, мы можем передать ей простой список с нашей партией, чтобы получить наши предсказания. По умолчанию она возвращает предсказания и цели, но поскольку у нас уже есть цели, мы можем эффективно игнорировать их, присвоив специальной переменной `_`:

In [ ]:
preds,_ = learn.get_preds(dl=[(x,y)])
preds[0]

Фактические предсказания — это 37 вероятностей между 0 и 1, которые в сумме дают 1:

In [ ]:
len(preds[0]),preds[0].sum()

Чтобы преобразовать активации нашей модели в такие предсказания, мы использовали функцию активации, называемую *softmax*.

### Softmax

В нашей модели классификации мы используем функцию активации softmax на последнем слое, чтобы гарантировать, что активации находятся в пределах от 0 до 1 и в сумме дают 1.

Softmax похожа на сигмоидную функцию, которую мы видели ранее. Напоминаем, что сигмоид выглядит так:

In [ ]:
plot_function(torch.sigmoid, min=-4,max=4)

Мы можем применить эту функцию к одному столбцу активаций из нейронной сети и получить обратно столбец чисел от 0 до 1, так что это очень полезная функция активации для нашего последнего слоя.

Теперь подумайте о том, что происходит, если мы хотим иметь больше категорий в нашей цели (например, наши 37 пород питомцев). Это означает, что нам нужно больше активаций, чем просто один столбец: нам нужна активация *для каждой категории*. Мы можем создать, например, нейронную сеть, которая предсказывает 3 и 7, и которая возвращает две активации, по одной для каждого класса — это будет хорошим первым шагом к созданию более общего подхода. Давайте просто используем случайные числа со стандартным отклонением 2 (поэтому мы умножаем `randn` на 2) для этого примера, предполагая, что у нас есть 6 изображений и 2 возможные категории (где первый столбец представляет 3, а второй — 7):

In [ ]:
#hide
torch.random.manual_seed(42);

In [ ]:
acts = torch.randn((6,2))*2
acts

Мы не можем просто взять сигмоиду от этого напрямую, поскольку мы не получаем строки, которые в сумме дают 1 (т.е. мы хотим, чтобы вероятность быть 3 плюс вероятность быть 7 в сумме давали 1):

In [ ]:
acts.sigmoid()

В \<\<chapter_mnist_basics\>\> наша нейронная сеть создала одну активацию на изображение, которую мы пропустили через функцию `sigmoid`. Эта единственная активация представляла уверенность модели в том, что входное значение — это 3. Двоичные задачи являются особым случаем задач классификации, поскольку цель может рассматриваться как одно булево значение, как мы делали в `mnist_loss`. Но двоичные задачи также можно рассматривать в контексте более общей группы классификаторов с любым количеством категорий: в данном случае у нас есть две категории. Как мы видели в классификаторе медведей, наша нейронная сеть будет возвращать одну активацию на категорию.

Так что же на самом деле означают эти активации в двоичном случае? Одна пара активаций просто указывает на *относительную* уверенность в том, что входное значение является 3 по сравнению с тем, что оно является 7. Общие значения, будь они высокими или низкими, не имеют значения — важно только, какое из них выше и на сколько.

Мы бы ожидали, что, поскольку это просто другой способ представления одной и той же задачи, мы сможем использовать `sigmoid` напрямую на версии нашей нейронной сети с двумя активациями. И действительно, мы можем! Мы можем просто взять *разницу* между активациями нейронной сети, потому что это отражает, насколько мы более уверены в том, что входное значение является 3, чем 7, а затем взять сигмоиду от этого:

In [ ]:
(acts[:,0]-acts[:,1]).sigmoid()

Второй столбец (вероятность того, что это 7) будет просто равен этому значению, вычтенному из 1. Теперь нам нужен способ сделать все это, который также работает для более чем двух столбцов. Оказывается, что эта функция, называемая `softmax`, именно такая:

```python
def softmax(x): return exp(x) / exp(x).sum(dim=1, keepdim=True)
```

> терминология: Экспоненциальная функция (exp): Буквально определена как `e**x`, где `e` — это специальное число, приблизительно равное 2.718. Это обратная функция натурального логарифма. Обратите внимание, что `exp` всегда положительна и увеличивается _очень_ быстро!

Давайте проверим, что `softmax` возвращает те же значения, что и `sigmoid` для первого столбца, а также те значения, вычтенные из 1, для второго столбца:

In [ ]:
sm_acts = torch.softmax(acts, dim=1)
sm_acts

`softmax` является многоклассовым эквивалентом `sigmoid` — его необходимо использовать всякий раз, когда у нас более двух категорий, и вероятности категорий должны суммироваться до 1. Мы часто используем его даже при наличии всего лишь двух категорий, чтобы сделать всё немного более последовательным. Мы могли бы создать другие функции, которые обладают свойствами, что все активации находятся в диапазоне от 0 до 1 и суммируются до 1; однако ни одна другая функция не имеет такой же связи с функцией `sigmoid`, которая, как мы видели, является гладкой и симметричной. Также мы вскоре увидим, что функция `softmax` хорошо работает в связке с функцией потерь, которую мы рассмотрим в следующем разделе.

Если у нас есть три активации на выходе, например, в нашем классификаторе медведей, то вычисление `softmax` для одного изображения медведя будет выглядеть примерно так \<\<bear_softmax\>\>.

<img alt="Bear softmax example" width="280" id="bear_softmax" caption="Example of softmax on the bear classifier" src="https://github.com/Wansmer/fastbook/blob/ru/images/att_00062.png?raw=1">

Что делает эта функция на практике? Использование экспоненты гарантирует, что все наши числа положительные, а деление на сумму обеспечивает, что у нас будет набор чисел, которые в сумме дают 1. Экспонента также обладает приятным свойством: если одно из чисел в наших активациях `x` немного больше остальных, экспонента усилит это (поскольку она растет, ну... экспоненциально), что означает, что в `softmax` это число будет ближе к 1.

Интуитивно, функция `softmax` *действительно* стремится выбрать один класс среди других, поэтому она идеальна для обучения классификатора, когда мы знаем, что каждое изображение имеет определенную метку. (Обратите внимание, что это может быть менее идеальным во время вывода, так как вы можете захотеть, чтобы ваша модель иногда сообщала вам, что она не распознает ни один из классов, которые она видела во время обучения, и не выбирала класс только потому, что у него немного больший балл активации. В этом случае может быть лучше обучить модель, используя несколько бинарных выходных колонок, каждая из которых использует активацию `sigmoid`.)

`Softmax` является первой частью функции потерь кросс-энтропии — второй частью является логарифмическая вероятность.

### Логарифмическое правдоподобие

Когда мы рассчитывали потерю для нашего примера MNIST в предыдущей главе, мы использовали:

```python
def mnist_loss(inputs, targets):
    inputs = inputs.sigmoid()
    return torch.where(targets==1, 1-inputs, inputs).mean()
```

Так же, как мы перешли от сигмоиды к softmax, нам нужно расширить функцию потерь, чтобы она работала не только с бинарной классификацией — она должна уметь классифицировать любое количество категорий (в данном случае у нас 37 категорий). Наши активации после softmax находятся в пределах от 0 до 1 и в сумме дают 1 для каждой строки в партии предсказаний. Наши цели — это целые числа от 0 до 36. Более того, потеря кросс-энтропии обобщает нашу потерю бинарной классификации и позволяет иметь более одной правильной метки на пример (что называется многометочной классификацией, которую мы обсудим в главе 6).

В бинарном случае мы использовали `torch.where`, чтобы выбрать между `inputs` и `1-inputs`. Когда мы рассматриваем бинарную классификацию как общую задачу классификации с двумя категориями, на самом деле это становится даже проще, потому что (как мы видели в предыдущем разделе) у нас теперь есть два столбца, содержащие эквиваленты `inputs` и `1-inputs`. Поскольку на каждый пример есть только одна правильная метка, все, что нам нужно сделать, это выбрать соответствующий столбец (вместо того, чтобы умножать несколько вероятностей). Давайте попробуем реализовать это в PyTorch. Для нашего синтетического примера с 3 и 7, предположим, что это наши метки:

In [ ]:
targ = tensor([0,1,0,1,1,0])

и это активации softmax:

In [ ]:
sm_acts

Затем для каждого элемента `targ` мы можем использовать это, чтобы выбрать соответствующий столбец из `sm_acts`, используя индексацию тензора, вот так:

In [ ]:
idx = range(6)
sm_acts[idx, targ]

Чтобы точно увидеть, что здесь происходит, давайте соберем все столбцы вместе в таблицу. Здесь первые два столбца — это наши активации, затем у нас есть цели и индекс строки. Мы объясним последний столбец, `result`, ниже:

In [ ]:
#hide_input
from IPython.display import HTML
df = pd.DataFrame(sm_acts, columns=["3","7"])
df['targ'] = targ
df['idx'] = idx
df['result'] = sm_acts[range(6), targ]
t = df.style.hide_index()
#To have html code compatible with our script
html = t._repr_html_().split('</style>')[1]
html = re.sub(r'<table id="([^"]+)"\s*>', r'<table >', html)
display(HTML(html))

Смотря на эту таблицу, вы можете увидеть, что столбец `result` можно вычислить, используя столбцы `targ` и `idx` в качестве индексов для двумерной матрицы, содержащей столбцы `3` и `7`. Именно это и делает `sm_acts[idx, targ]`. Действительно интересная вещь здесь заключается в том, что это работает так же хорошо и с более чем двумя столбцами. Чтобы увидеть это, рассмотрим, что произойдет, если мы добавим столбец активации для каждой цифры (от 0 до 9), и тогда `targ` будет содержать число от 0 до 9.

PyTorch предоставляет функцию, которая делает именно то же самое, что и `sm_acts[range(n), targ]` (за исключением того, что она берет отрицательное значение, потому что при последующем применении логарифма у нас будут отрицательные числа), называемую `nll_loss` (*NLL* означает *отрицательное логарифмическое правдоподобие*):

In [ ]:
-sm_acts[idx, targ]

In [ ]:
F.nll_loss(sm_acts, targ, reduction='none')

Несмотря на свое название, эта функция PyTorch не берет логарифм. Мы увидим, почему в следующем разделе, но сначала давайте рассмотрим, почему взятие логарифма может быть полезным.

> предупреждение: Запутывающее название, будьте осторожны: nll в `nll_loss` означает "отрицательное логарифмическое правдоподобие", но на самом деле он вообще не берет логарифм! Он предполагает, что вы _уже_ взяли логарифм. В PyTorch есть функция, называемая `log_softmax`, которая сочетает `log` и `softmax` быстрым и точным способом. `nll_loss` предназначен для использования после `log_softmax`.

#### Taking the Log

Recall that cross entropy loss may involve the multiplication of many numbers.  Multiplying lots of negative numbers together can cause problems like [numerical underflow](https://en.wikipedia.org/wiki/Arithmetic_underflow) in computers.  Therefore, we want to transform these probabilities to larger values so we can perform mathematical operations on them.  There is a mathematical function that does exactly this: the *logarithm* (available as `torch.log`). It is not defined for numbers less than 0, and looks like this between 0 and 1:

In [ ]:
plot_function(torch.log, min=0,max=1, ty='log(x)', tx='x')

Additionally, we want to ensure our model is able to detect differences between small numbers.  For example, consider the probabilities of .01 and .001.  Indeed, those numbers are very close together—but in another sense, 0.01 is 10 times more confident than 0.001.  By taking the log of our probabilities, we prevent these important differences from being ignored.

Does "logarithm" ring a bell? The logarithm function has this identity:

```
y = b**a
a = log(y,b)
```

In this case, we're assuming that `log(y,b)` returns *log y base b*. However, PyTorch actually doesn't define `log` this way: `log` in Python uses the special number `e` (2.718...) as the base.

Perhaps a logarithm is something that you have not thought about for the last 20 years or so. But it's a mathematical idea that is going to be really critical for many things in deep learning, so now would be a great time to refresh your memory. The key thing to know about logarithms is this relationship:

    log(a*b) = log(a)+log(b)

When we see it in that format, it looks a bit boring; but think about what this really means. It means that logarithms increase linearly when the underlying signal increases exponentially or multiplicatively. This is used, for instance, in the Richter scale of earthquake severity, and the dB scale of noise levels. It's also often used on financial charts, where we want to show compound growth rates more clearly. Computer scientists love using logarithms, because it means that multiplication, which can create really really large and really really small numbers, can be replaced by addition, which is much less likely to result in scales that are difficult for our computers to handle.

Observe that the log of a number approaches negative infinity as the number approaches zero.  In our case, since the result relfects the predicted probability of the correct label, we want our loss function to return a small value when the prediction is "good" (closer to 1) and a large value when the prediction is "bad" (closer to 0).  We can achieve this by taking the negative of the log:

In [ ]:
plot_function(lambda x: -1*torch.log(x), min=0,max=1, tx='x', ty='- log(x)', title = 'Log Loss when true label = 1')

> s: It's not just computer scientists that love logs! Until computers came along, engineers and scientists used a special ruler called a "slide rule" that did multiplication by adding logarithms. Logarithms are widely used in physics, for multiplying very big or very small numbers, and many other fields.

Let's go ahead and update our previous table with an additional column, `loss` to reflect this loss function:

In [ ]:
#hide_input
from IPython.display import HTML
df['loss'] = -torch.log(tensor(df['result']))
t = df.style.hide_index()
#To have html code compatible with our script
html = t._repr_html_().split('</style>')[1]
html = re.sub(r'<table id="([^"]+)"\s*>', r'<table >', html)
display(HTML(html))

Notice how the loss is very large in the third and fourth rows where the predictions are confident and wrong, or in other words have high probabilities on the wrong class.  One benefit of using the log to calculate the loss is that our loss function penalizes predictions that are both confident and wrong.  This kind of penalty works well in practice to aid in more effective model training.  

> s: There are other loss functions such as [focal loss](https://arxiv.org/pdf/1708.02002.pdf) that allow you control this penalty with a parameter.  We do not discuss that loss function in this book.

We're calculating the loss from the column containing the correct label. Because there is only one "right" answer per example, we don't need to consider the other columns, because by the definition of softmax, they add up to 1 minus the activation corresponding to the correct label. As long as the activation columns sum to 1 (as they will, if we use softmax), then we'll have a loss function that shows how well we're predicting each digit.  Therefore, making the activation for the correct label as high as possible must mean we're also decreasing the activations of the remaining columns.  

### Negative Log Likelihood

Taking the mean of the negative log of our probabilities (taking the mean of the `loss` column of our table) gives us the *negative log likelihood* loss, which is another name for cross-entropy loss. Recall that PyTorch's `nll_loss` assumes that you already took the log of the softmax, so it doesn't actually do the logarithm for you.

When we first take the softmax, and then the log likelihood of that, that combination is called *cross-entropy loss*. In PyTorch, this is available as `nn.CrossEntropyLoss` (which, in practice, actually does `log_softmax` and then `nll_loss`):

In [ ]:
loss_func = nn.CrossEntropyLoss()

As you see, this is a class. Instantiating it gives you an object which behaves like a function:

In [ ]:
loss_func(acts, targ)

All PyTorch loss functions are provided in two forms, the class just shown above, and also a plain functional form, available in the `F` namespace:

In [ ]:
F.cross_entropy(acts, targ)

Either one works fine and can be used in any situation. We've noticed that most people tend to use the class version, and that's more often used in PyTorch's official docs and examples, so we'll tend to use that too.

By default PyTorch loss functions take the mean of the loss of all items. You can use `reduction='none'` to disable that:

In [ ]:
nn.CrossEntropyLoss(reduction='none')(acts, targ)

You will notice these values match the `loss` column in our table exactly.

> s: An interesting feature about cross-entropy loss appears when we consider its gradient. The gradient of `cross_entropy(a,b)` is just `softmax(a)-b`. Since `softmax(a)` is just the final activation of the model, that means that the gradient is proportional to the difference between the prediction and the target. This is the same as mean squared error in regression (assuming there's no final activation function such as that added by `y_range`), since the gradient of `(a-b)**2` is `2*(a-b)`. Because the gradient is linear, that means we won't see sudden jumps or exponential increases in gradients, which should lead to smoother training of models.

We have now seen all the pieces hidden behind our loss function. But while this puts a number on how well (or badly) our model is doing, it does nothing to help us know if it's actually any good. Let's now see some ways to interpret our model's predictions.

## Model Interpretation

It's very hard to interpret loss functions directly, because they are designed to be things computers can differentiate and optimize, not things that people can understand. That's why we have metrics. These are not used in the optimization process, but just to help us poor humans understand what's going on. In this case, our accuracy is looking pretty good already! So where are we making mistakes?

We saw in <<chapter_intro>> that we can use a confusion matrix to see where our model is doing well, and where it's doing badly:

In [ ]:
#width 600
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(12,12), dpi=60)

Oh dear—in this case, a confusion matrix is very hard to read. We have 37 different breeds of pet, which means we have 37×37 entries in this giant matrix! Instead, we can use the `most_confused` method, which just shows us the cells of the confusion matrix with the most incorrect predictions (here, with at least 5 or more):

In [ ]:
interp.most_confused(min_val=5)

Since we are not pet breed experts, it is hard for us to know whether these category errors reflect actual difficulties in recognizing breeds. So again, we turn to Google. A little bit of Googling tells us that the most common category errors shown here are actually breed differences that even expert breeders sometimes disagree about. So this gives us some comfort that we are on the right track.

We seem to have a good baseline. What can we do now to make it even better?

## Improving Our Model

We will now look at a range of techniques to improve the training of our model and make it better. While doing so, we will explain a little bit more about transfer learning and how to fine-tune our pretrained model as best as possible, without breaking the pretrained weights.

The first thing we need to set when training a model is the learning rate. We saw in the previous chapter that it needs to be just right to train as efficiently as possible, so how do we pick a good one? fastai provides a tool for this.

### The Learning Rate Finder

One of the most important things we can do when training a model is to make sure that we have the right learning rate. If our learning rate is too low, it can take many, many epochs to train our model. Not only does this waste time, but it also means that we may have problems with overfitting, because every time we do a complete pass through the data, we give our model a chance to memorize it.

So let's just make our learning rate really high, right? Sure, let's try that and see what happens:

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(1, base_lr=0.1)

That doesn't look good. Here's what happened. The optimizer stepped in the correct direction, but it stepped so far that it totally overshot the minimum loss. Repeating that multiple times makes it get further and further away, not closer and closer!

What do we do to find the perfect learning rate—not too high, and not too low? In 2015 the researcher Leslie Smith came up with a brilliant idea, called the *learning rate finder*. His idea was to start with a very, very small learning rate, something so small that we would never expect it to be too big to handle. We use that for one mini-batch, find what the losses are afterwards, and then increase the learning rate by some percentage (e.g., doubling it each time). Then we do another mini-batch, track the loss, and double the learning rate again. We keep doing this until the loss gets worse, instead of better. This is the point where we know we have gone too far. We then select a learning rate a bit lower than this point. Our advice is to pick either:

- One order of magnitude less than where the minimum loss was achieved (i.e., the minimum divided by 10)
- The last point where the loss was clearly decreasing

The learning rate finder computes those points on the curve to help you. Both these rules usually give around the same value. In the first chapter, we didn't specify a learning rate, using the default value from the fastai library (which is 1e-3):

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
lr_min,lr_steep = learn.lr_find(suggest_funcs=(minimum, steep))

In [ ]:
print(f"Minimum/10: {lr_min:.2e}, steepest point: {lr_steep:.2e}")

We can see on this plot that in the range 1e-6 to 1e-3, nothing really happens and the model doesn't train. Then the loss starts to decrease until it reaches a minimum, and then increases again. We don't want a learning rate greater than 1e-1 as it will give a training that diverges like the one before (you can try for yourself), but 1e-1 is already too high: at this stage we've left the period where the loss was decreasing steadily.

In this learning rate plot it appears that a learning rate around 3e-3 would be appropriate, so let's choose that:

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fine_tune(2, base_lr=3e-3)

> Note: Logarithmic Scale: The learning rate finder plot has a logarithmic scale, which is why the middle point between 1e-3 and 1e-2 is between 3e-3 and 4e-3. This is because we care mostly about the order of magnitude of the learning rate.

It's interesting that the learning rate finder was only discovered in 2015, while neural networks have been under development since the 1950s. Throughout that time finding a good learning rate has been, perhaps, the most important and challenging issue for practitioners. The solution does not require any advanced maths, giant computing resources, huge datasets, or anything else that would make it inaccessible to any curious researcher. Furthermore, Leslie Smith, was not part of some exclusive Silicon Valley lab, but was working as a naval researcher. All of this is to say: breakthrough work in deep learning absolutely does not require access to vast resources, elite teams, or advanced mathematical ideas. There is lots of work still to be done that requires just a bit of common sense, creativity, and tenacity.

Now that we have a good learning rate to train our model, let's look at how we can fine-tune the weights of a pretrained model.

### Unfreezing and Transfer Learning

We discussed briefly in <<chapter_intro>> how transfer learning works. We saw that the basic idea is that a pretrained model, trained potentially on millions of data points (such as ImageNet), is fine-tuned for some other task. But what does this really mean?

We now know that a convolutional neural network consists of many linear layers with a nonlinear activation function between each pair, followed by one or more final linear layers with an activation function such as softmax at the very end. The final linear layer uses a matrix with enough columns such that the output size is the same as the number of classes in our model (assuming that we are doing classification).

This final linear layer is unlikely to be of any use for us when we are fine-tuning in a transfer learning setting, because it is specifically designed to classify the categories in the original pretraining dataset. So when we do transfer learning we remove it, throw it away, and replace it with a new linear layer with the correct number of outputs for our desired task (in this case, there would be 37 activations).

This newly added linear layer will have entirely random weights. Therefore, our model prior to fine-tuning has entirely random outputs. But that does not mean that it is an entirely random model! All of the layers prior to the last one have been carefully trained to be good at image classification tasks in general. As we saw in the images from the [Zeiler and Fergus paper](https://arxiv.org/pdf/1311.2901.pdf) in <<chapter_intro>> (see <<img_layer1>> through <<img_layer4>>), the first few layers encode very general concepts, such as finding gradients and edges, and later layers encode concepts that are still very useful for us, such as finding eyeballs and fur.

We want to train a model in such a way that we allow it to remember all of these generally useful ideas from the pretrained model, use them to solve our particular task (classify pet breeds), and only adjust them as required for the specifics of our particular task.

Our challenge when fine-tuning is to replace the random weights in our added linear layers with weights that correctly achieve our desired task (classifying pet breeds) without breaking the carefully pretrained weights and the other layers. There is actually a very simple trick to allow this to happen: tell the optimizer to only update the weights in those randomly added final layers. Don't change the weights in the rest of the neural network at all. This is called *freezing* those pretrained layers.

When we create a model from a pretrained network fastai automatically freezes all of the pretrained layers for us. When we call the `fine_tune` method fastai does two things:

- Trains the randomly added layers for one epoch, with all other layers frozen
- Unfreezes all of the layers, and trains them all for the number of epochs requested

Although this is a reasonable default approach, it is likely that for your particular dataset you may get better results by doing things slightly differently. The `fine_tune` method has a number of parameters you can use to change its behavior, but it might be easiest for you to just call the underlying methods directly if you want to get some custom behavior. Remember that you can see the source code for the method by using the following syntax:

    learn.fine_tune??

So let's try doing this manually ourselves. First of all we will train the randomly added layers for three epochs, using `fit_one_cycle`. As mentioned in <<chapter_intro>>, `fit_one_cycle` is the suggested way to train models without using `fine_tune`. We'll see why later in the book; in short, what `fit_one_cycle` does is to start training at a low learning rate, gradually increase it for the first section of training, and then gradually decrease it again for the last section of training.

In [ ]:
learn.fine_tune??

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fit_one_cycle(3, 3e-3)

Then we'll unfreeze the model:

In [ ]:
learn.unfreeze()

and run `lr_find` again, because having more layers to train, and weights that have already been trained for three epochs, means our previously found learning rate isn't appropriate any more:

In [ ]:
learn.lr_find()

Note that the graph is a little different from when we had random weights: we don't have that sharp descent that indicates the model is training. That's because our model has been trained already. Here we have a somewhat flat area before a sharp increase, and we should take a point well before that sharp increase—for instance, 1e-5. The point with the maximum gradient isn't what we look for here and should be ignored.

Let's train at a suitable learning rate:

In [ ]:
learn.fit_one_cycle(6, lr_max=1e-5)

This has improved our model a bit, but there's more we can do. The deepest layers of our pretrained model might not need as high a learning rate as the last ones, so we should probably use different learning rates for those—this is known as using *discriminative learning rates*.

### Discriminative Learning Rates

Even after we unfreeze, we still care a lot about the quality of those pretrained weights. We would not expect that the best learning rate for those pretrained parameters would be as high as for the randomly added parameters, even after we have tuned those randomly added parameters for a few epochs. Remember, the pretrained weights have been trained for hundreds of epochs, on millions of images.

In addition, do you remember the images we saw in <<chapter_intro>>, showing what each layer learns? The first layer learns very simple foundations, like edge and gradient detectors; these are likely to be just as useful for nearly any task. The later layers learn much more complex concepts, like "eye" and "sunset," which might not be useful in your task at all (maybe you're classifying car models, for instance). So it makes sense to let the later layers fine-tune more quickly than earlier layers.

Therefore, fastai's default approach is to use discriminative learning rates. This was originally developed in the ULMFiT approach to NLP transfer learning that we will introduce in <<chapter_nlp>>. Like many good ideas in deep learning, it is extremely simple: use a lower learning rate for the early layers of the neural network, and a higher learning rate for the later layers (and especially the randomly added layers). The idea is based on insights developed by [Jason Yosinski](https://arxiv.org/abs/1411.1792), who showed in 2014 that with transfer learning different layers of a neural network should train at different speeds, as seen in <<yosinski>>.

<img alt="Impact of different layers and training methods on transfer learning (Yosinski)" width="680" caption="Impact of different layers and training methods on transfer learning (courtesy of Jason Yosinski et al.)" id="yosinski" src="https://github.com/Wansmer/fastbook/blob/ru/images/att_00039.png?raw=1">

fastai lets you pass a Python `slice` object anywhere that a learning rate is expected. The first value passed will be the learning rate in the earliest layer of the neural network, and the second value will be the learning rate in the final layer. The layers in between will have learning rates that are multiplicatively equidistant throughout that range. Let's use this approach to replicate the previous training, but this time we'll only set the *lowest* layer of our net to a learning rate of 1e-6; the other layers will scale up to 1e-4. Let's train for a while and see what happens:

In [ ]:
learn = vision_learner(dls, resnet34, metrics=error_rate)
learn.fit_one_cycle(3, 3e-3)
learn.unfreeze()
learn.fit_one_cycle(12, lr_max=slice(1e-6,1e-4))

Now the fine-tuning is working great!

fastai can show us a graph of the training and validation loss:

In [ ]:
learn.recorder.plot_loss()

As you can see, the training loss keeps getting better and better. But notice that eventually the validation loss improvement slows, and sometimes even gets worse! This is the point at which the model is starting to over fit. In particular, the model is becoming overconfident of its predictions. But this does *not* mean that it is getting less accurate, necessarily. Take a look at the table of training results per epoch, and you will often see that the accuracy continues improving, even as the validation loss gets worse. In the end what matters is your accuracy, or more generally your chosen metrics, not the loss. The loss is just the function we've given the computer to help us to optimize.

Another decision you have to make when training the model is for how long to train for. We'll consider that next.

### Selecting the Number of Epochs

Often you will find that you are limited by time, rather than generalization and accuracy, when choosing how many epochs to train for. So your first approach to training should be to simply pick a number of epochs that will train in the amount of time that you are happy to wait for. Then look at the training and validation loss plots, as shown above, and in particular your metrics, and if you see that they are still getting better even in your final epochs, then you know that you have not trained for too long.

On the other hand, you may well see that the metrics you have chosen are really getting worse at the end of training. Remember, it's not just that we're looking for the validation loss to get worse, but the actual metrics. Your validation loss will first get worse during training because the model gets overconfident, and only later will get worse because it is incorrectly memorizing the data. We only care in practice about the latter issue. Remember, our loss function is just something that we use to allow our optimizer to have something it can differentiate and optimize; it's not actually the thing we care about in practice.

Before the days of 1cycle training it was very common to save the model at the end of each epoch, and then select whichever model had the best accuracy out of all of the models saved in each epoch. This is known as *early stopping*. However, this is very unlikely to give you the best answer, because those epochs in the middle occur before the learning rate has had a chance to reach the small values, where it can really find the best result. Therefore, if you find that you have overfit, what you should actually do is retrain your model from scratch, and this time select a total number of epochs based on where your previous best results were found.

If you have the time to train for more epochs, you may want to instead use that time to train more parameters—that is, use a deeper architecture.

### Deeper Architectures

In general, a model with more parameters can model your data more accurately. (There are lots and lots of caveats to this generalization, and it depends on the specifics of the architectures you are using, but it is a reasonable rule of thumb for now.) For most of the architectures that we will be seeing in this book, you can create larger versions of them by simply adding more layers. However, since we want to use pretrained models, we need to make sure that we choose a number of layers that have already been pretrained for us.

This is why, in practice, architectures tend to come in a small number of variants. For instance, the ResNet architecture that we are using in this chapter comes in variants with 18, 34, 50, 101, and 152 layer, pretrained on ImageNet. A larger (more layers and parameters; sometimes described as the "capacity" of a model) version of a ResNet will always be able to give us a better training loss, but it can suffer more from overfitting, because it has more parameters to overfit with.

In general, a bigger model has the ability to better capture the real underlying relationships in your data, and also to capture and memorize the specific details of your individual images.

However, using a deeper model is going to require more GPU RAM, so you may need to lower the size of your batches to avoid an *out-of-memory error*. This happens when you try to fit too much inside your GPU and looks like:

```
Cuda runtime error: out of memory
```

You may have to restart your notebook when this happens. The way to solve it is to use a smaller batch size, which means passing smaller groups of images at any given time through your model. You can pass the batch size you want to the call creating your `DataLoaders` with `bs=`.

The other downside of deeper architectures is that they take quite a bit longer to train. One technique that can speed things up a lot is *mixed-precision training*. This refers to using less-precise numbers (*half-precision floating point*, also called *fp16*) where possible during training. As we are writing these words in early 2020, nearly all current NVIDIA GPUs support a special feature called *tensor cores* that can dramatically speed up neural network training, by 2-3x. They also require a lot less GPU memory. To enable this feature in fastai, just add `to_fp16()` after your `Learner` creation (you also need to import the module).

You can't really know ahead of time what the best architecture for your particular problem is—you need to try training some. So let's try a ResNet-50 now with mixed precision:

In [ ]:
from fastai.callback.fp16 import *
learn = vision_learner(dls, resnet50, metrics=error_rate).to_fp16()
learn.fine_tune(6, freeze_epochs=3)

You'll see here we've gone back to using `fine_tune`, since it's so handy! We can pass `freeze_epochs` to tell fastai how many epochs to train for while frozen. It will automatically change learning rates appropriately for most datasets.

In this case, we're not seeing a clear win from the deeper model. This is useful to remember—bigger models aren't necessarily better models for your particular case! Make sure you try small models before you start scaling up.

## Conclusion

In this chapter you learned some important practical tips, both for getting your image data ready for modeling (presizing, data block summary) and for fitting the model (learning rate finder, unfreezing, discriminative learning rates, setting the number of epochs, and using deeper architectures). Using these tools will help you to build more accurate image models, more quickly.

We also discussed cross-entropy loss. This part of the book is worth spending plenty of time on. You aren't likely to need to actually implement cross-entropy loss from scratch yourself in practice, but it's really important you understand the inputs to and output from that function, because it (or a variant of it, as we'll see in the next chapter) is used in nearly every classification model. So when you want to debug a model, or put a model in production, or improve the accuracy of a model, you're going to need to be able to look at its activations and loss, and understand what's going on, and why. You can't do that properly if you don't understand your loss function.

If cross-entropy loss hasn't "clicked" for you just yet, don't worry—you'll get there! First, go back to the last chapter and make sure you really understand `mnist_loss`. Then work gradually through the cells of the notebook for this chapter, where we step through each piece of cross-entropy loss. Make sure you understand what each calculation is doing, and why. Try creating some small tensors yourself and pass them into the functions, to see what they return.

Remember: the choices made in the implementation of cross-entropy loss are not the only possible choices that could have been made. Just like when we looked at regression we could choose between mean squared error and mean absolute difference (L1). If you have other ideas for possible functions that you think might work, feel free to give them a try in this chapter's notebook! (Fair warning though: you'll probably find that the model will be slower to train, and less accurate. That's because the gradient of cross-entropy loss is proportional to the difference between the activation and the target, so SGD always gets a nicely scaled step for the weights.)

## Questionnaire

1. Why do we first resize to a large size on the CPU, and then to a smaller size on the GPU?
1. If you are not familiar with regular expressions, find a regular expression tutorial, and some problem sets, and complete them. Have a look on the book's website for suggestions.
1. What are the two ways in which data is most commonly provided, for most deep learning datasets?
1. Look up the documentation for `L` and try using a few of the new methods that it adds.
1. Look up the documentation for the Python `pathlib` module and try using a few methods of the `Path` class.
1. Give two examples of ways that image transformations can degrade the quality of the data.
1. What method does fastai provide to view the data in a `DataLoaders`?
1. What method does fastai provide to help you debug a `DataBlock`?
1. Should you hold off on training a model until you have thoroughly cleaned your data?
1. What are the two pieces that are combined into cross-entropy loss in PyTorch?
1. What are the two properties of activations that softmax ensures? Why is this important?
1. When might you want your activations to not have these two properties?
1. Calculate the `exp` and `softmax` columns of <<bear_softmax>> yourself (i.e., in a spreadsheet, with a calculator, or in a notebook).
1. Why can't we use `torch.where` to create a loss function for datasets where our label can have more than two categories?
1. What is the value of log(-2)? Why?
1. What are two good rules of thumb for picking a learning rate from the learning rate finder?
1. What two steps does the `fine_tune` method do?
1. In Jupyter Notebook, how do you get the source code for a method or function?
1. What are discriminative learning rates?
1. How is a Python `slice` object interpreted when passed as a learning rate to fastai?
1. Why is early stopping a poor choice when using 1cycle training?
1. What is the difference between `resnet50` and `resnet101`?
1. What does `to_fp16` do?

### Further Research

1. Find the paper by Leslie Smith that introduced the learning rate finder, and read it.
1. See if you can improve the accuracy of the classifier in this chapter. What's the best accuracy you can achieve? Look on the forums and the book's website to see what other students have achieved with this dataset, and how they did it.